In [ ]:
from flask import Flask, request, Response
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
import matplotlib.pyplot as plt
from multiprocessing import Process
import time
import pandas
import math

In [ ]:
app = Flask("app")

ab = [1, 1]
total = [2, 2]
ucb = [10, 10]
cookie = "1"

@app.route("/")
def home():
    global ab, total, ucb, cookie
    if ucb[0] >= ucb[1]:
        total[0] = total[0] + 1
        resp = Response('Cat! <a href="visit?from=cat">Visit</a>')
    else:
        total[1] = total[1] + 1
        resp = Response('Dog! <a href="visit?from=dog">Visit</a>')
    resp.set_cookie("visit", "0")
    return resp

@app.route("/visit")
def test():
    global ab, total, ucb, cookie
    cookie = request.cookies.get("visit", "0")
    if cookie == "0" and request.args:
        if request.args["from"] == "cat":
            ab[0] = ab[0] + 1
            ucb[0] = ab[0] / total[0] + math.sqrt((2 * math.log(total[0] + total[1]) / total[0]))
        else:
            ab[1] = ab[1] + 1
            ucb[1] = ab[1] / total[1] + math.sqrt((2 * math.log(total[0] + total[1]) / total[1]))
    df = pandas.DataFrame([[ab[0] / total[0], ucb[0]], [ab[1] / total[1], ucb[1]]], columns = ["CRT", "UCB"], index = ["cat", "dog"])
    resp = Response(df.to_html())
    resp.set_cookie("visit", "1")
    return resp

app.run(host = "0.0.0.0", debug = False, threaded = False)